# Searching for the category

For this code along we are only going to use the products DataFrame. However, if you believe there is information in other tables that can help to create categories, please feel free to explore.

In [ ]:
import pandas as pd

In [ ]:
# products_cl.csv
url = "https://drive.google.com/file/d/1IPnKYLSnrST0HBZMeSeNrhFfWU31QvwB/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_cl = pd.read_csv(path)

In [ ]:
product_category_df = products_cl.copy()

In [ ]:
product_category_df.head()

,sku,name,desc,price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364


In [ ]:
product_category_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9992 entries, 0 to 9991
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9992 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 468.5+ KB


## 1.&nbsp; Category creation by search term
Let's start by creating a column `category`. For now we'll fill this column with a blank string `""`.

In [ ]:
product_category_df["category"] = ""
product_category_df.head()

,sku,name,desc,price,in_stock,type,category
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696,
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401,
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387,
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230,
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364,


We can find all the products with certain words in their `description` using `.loc[]` and `.str.contains()`. Here we'll look at all the items that have the word `keyboard` in their description.

In [ ]:
product_category_df.loc[product_category_df["desc"].str.contains("keyboard", case=False)]

,sku,name,desc,price,in_stock,type,category
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401,
15,MOS0021,Clearguard Moshi MacBook Pro and Air,Keyboard Protector MacBook Pro 13-inch Retina ...,24.95,0,13835403,
24,APP0277,Apple Wireless Keyboard Keyboard (OEM) Mac,Ultrathin keyboard Apple Bluetooth Spanish (un...,79.00,0,13855401,
64,HGD0012,Henge Docks Click keyboard support iMac,Base to hold the Apple Magic TrackPad and Wire...,29.00,0,8696,
365,LOG0084,Logitech Ultrathin Keyboard Cover Keyboard Cov...,Ultrathin cover and cover with Bluetooth keybo...,89.99,0,12575403,
...,...,...,...,...,...,...,...
9720,PAC2508,Replacement Magic Wireless Keyboard by Matias ...,Keyboard replacement service at the time of pu...,119.99,1,13855401,
9751,MTF0008,Mistify Clean Screens Natural 500ml.,Spray cleaning screens and keyboards.,14.99,1,12085400,
9796,ZAG0026-A,Open - Zagg Rugged Keyboard Folio iPad Messeng...,Case reconditioned keyboard and adjustable pos...,99.99,0,12575403,
9932,APP1472,Apple Magic Keyboard English International,English keyboard Mac and Apple iPad Ultrathin ...,119.00,1,13855401,


Next, we change the value in the category column to `keyboard` for all of these keyboard products.

In [ ]:
product_category_df.loc[product_category_df["desc"].str.contains("keyboard", case=False), "category"] = "keyboard"

Let's take a look at the effect that had on the `category` column.

In [ ]:
product_category_df["category"].value_counts()

,count
category,
,9903
keyboard,89


## 2.&nbsp; Category creation using regex
We can also use a product's `name` to select products for our categories.

In [ ]:
product_category_df.loc[product_category_df["name"].str.contains("apple iphone", case=False)]

,sku,name,desc,price,in_stock,type,category
35,APP0308,AV Cable Adapter Apple iPhone iPad and iPod white,IPhone iPad iPod adapter and AV cable.,45.00,0,1230,
214,REP0100,Color change to White Apple iPhone 4,It is including parts and labor..,94.21,0,"1,44E+11",
215,REP0052,Color change to White Apple iPhone 4,It is including parts and labor..,94.21,0,"1,44E+11",
579,APP0675,Apple iPhone 5S 32GB Space Gray,New Free iPhone 5S 32GB (ME435Y / A).,559.00,0,NaN,
956,APP0823,Apple iPhone 6 16GB Silver,New iPhone 6 16GB Free (MG482QL / A).,639.00,0,NaN,
...,...,...,...,...,...,...,...
9790,AP20455,Like new - Apple iPhone 8 256GB Gold,Apple iPhone 8 reconditioned 256GB in Gold rea...,979.00,0,113291716,
9794,APP2482-A,Open - Apple iPhone 8 Plus 256GB Gold,Refurbished Apple iPhone 8 Plus 256GB Free Gold,1089.00,0,113281716,
9929,APP2477-A,Open - Apple iPhone 8 Plus 64GB Space Gray,Apple iPhone 8 Plus 64GB Space Gray,919.00,0,113281716,
9958,AP20467,Like new - Apple iPhone Silicone Case Cover 7 ...,Reconditioned silicone sleeve microfiber Apple...,45.00,0,11865403,


Looks like we get a lot of accessories included in this search. We can refine this using a little regex. Here, we will add `.{0,7}` at the beginning of the search: this means we will find all `apple iphone`s that have 7 or less characters preceding the term "apple iphone" - if there's 8 characters preceding the search term, it won't be found. This should help refine our search by using the nomenclature of the DataFrame to our advantage.

If you feel unsure about regex, please use [regex101](https://regex101.com/). It's really useful for checking your code, and parts of other people's code that you're unsure about.

In [ ]:
product_category_df.loc[product_category_df["name"].str.contains("^.{0,7}apple iphone", case=False)]

,sku,name,desc,price,in_stock,type,category
579,APP0675,Apple iPhone 5S 32GB Space Gray,New Free iPhone 5S 32GB (ME435Y / A).,559.0,0,NaN,
956,APP0823,Apple iPhone 6 16GB Silver,New iPhone 6 16GB Free (MG482QL / A).,639.0,0,NaN,
961,APP0829,Apple iPhone 6 Plus 16GB Silver,New iPhone 6 Plus 16G Free (MGA92QL / A).,749.0,0,NaN,
962,APP0822,Apple iPhone 6 16GB Space Gray,New iPhone 6 16GB Free (MG472QL / A).,639.0,0,NaN,
963,APP0825,Apple iPhone 6 64GB Space Gray,New iPhone 6 64GB Free (MG4F2QL / A).,749.0,0,NaN,
...,...,...,...,...,...,...,...
9585,APP1634-A,Open - Apple iPhone 7 Plus 32GB Black,New 32GB Apple iPhone 7 Plus Free Black,779.0,0,85651716,
9587,APP2540-A,Open - Apple iPhone Leather Folio X Baya,Leather case with box and official cover Apple,109.0,0,11865403,
9714,APP2562-A,Open - Apple iPhone Leather Case Cover Red,Reconditioned skin sheath official Apple desig...,45.0,0,11865403,
9794,APP2482-A,Open - Apple iPhone 8 Plus 256GB Gold,Refurbished Apple iPhone 8 Plus 256GB Free Gold,1089.0,0,113281716,


Now we can use the same trick as before to set the category - selecting the `category` column and setting it to the string of our choice.

In [ ]:
product_category_df.loc[product_category_df["name"].str.contains("^.{0,7}apple iphone", case=False), "category"] = "smartphone"

In [ ]:
product_category_df["category"].value_counts()

,count
category,
,9634
smartphone,269
keyboard,89


## 3.&nbsp; One product with multiple categories
A product may fit into multiple categories. To help us create multiple categories for one product, we will use the python addition assignment `+=`. The addition assignment is a shorthand way to add something (number, string, etc...) to a variable without changing the variable name.

Let's have a look at a couple of examples.

In [ ]:
a = 10
a = a + 5
a

15

In [ ]:
a = 10
a += 5
a

15

In [ ]:
b = "Tyrannosaurus"
b = b + " rex"
b

'Tyrannosaurus rex'

In [ ]:
b = "Tyrannosaurus"
b += " rex"
b

'Tyrannosaurus rex'

Now let's look at how this can help us in our category creation.

First, we'll reset all the values in the category column to an empty string `""`.

In [ ]:
product_category_df["category"] = ""

Now, let's create some categories and utilise the addition assignment.

In [ ]:
product_category_df.loc[product_category_df["desc"].str.contains("keyboard", case=False), "category"] += ", keyboard"
product_category_df.loc[product_category_df["name"].str.contains("^.{0,7}apple iphone", case=False), "category"] += ", smartphone"
product_category_df.loc[product_category_df["name"].str.contains("^.{0,7}apple ipod", case=False), "category"] += ", ipod"
product_category_df.loc[product_category_df["name"].str.contains("^.{0,7}apple ipad|tablet", case=False), "category"] += ", tablet"
product_category_df.loc[product_category_df["name"].str.contains("imac|mac mini|mac pro", case=False), "category"] += ", desktop"

In [ ]:
product_category_df["category"].value_counts()

,count
category,
,8362
", desktop",923
", tablet",307
", smartphone",269
", keyboard",83
", ipod",42
", keyboard, tablet",4
", keyboard, desktop",2


As you can see, some products now have 2 categories instead of just one. At the end, you can use your skills with string to tidy up the opening comma and space in the `category` column.

# Challenge. Your categories
Now it's your turn. We'll reset the Dataframe so that no categories exist, and it's up to you to create the categories based on keywords in the name and description. Feel free to go wild and make as many categories as you like.
* Remember you can also use regex to refine your searches.
* Remember you can use the or operator `|` to search for multiple terms at once.
* Remember to tidy up any untidy strings at the end.

In [ ]:
# resetting to original import
product_category_df = products_cl.copy()

In [ ]:
# your code here

## 4.&nbsp; [BONUS] Using `type` to create categories
There could be another way to create categories, but this one you'll have to explore this one alone.

We have the mysterious column `type` in the `products` table. This could potentially be ready-made categories labelled with numbers instead of words. Let's investigate.

In [ ]:
category_type_df = products_cl.copy()

Here are the `type`s that have the most products.

In [ ]:
category_type_df.groupby("type").count().nlargest(10, "sku")

,sku,name,desc,price,in_stock
type,,,,,
11865403,1057,1057,1057,1057,1057
12175397,939,939,939,939,939
1298,783,783,783,783,783
11935397,562,562,562,562,562
11905404,454,454,454,454,454
1282,373,373,373,373,373
12635403,362,362,362,362,362
13835403,269,269,269,269,269
"5,74E+15",247,247,247,247,247


Let's have a look at the first `type` to see if we can make categories from this column.

In [ ]:
category_type_df.loc[category_type_df["type"] == "11865403"].sample(10)

,sku,name,desc,price,in_stock,type
4585,TWS0099,Twelve South BookBook Case for iPhone 6 / 6S B...,card holder book cover seal with iPhone 6 / 6S,69.99,0,11865403
5131,UAG0045,Urban Armor Gear Case iPhone Pathfinder 7 Rust...,resistant cover for iPhone bumps and falls in ...,29.95,0,11865403
1013,TUC0175,Tucano Fabric Case iPhone 6 Blue,Cover for iPhone 6 ultra light polycarbonate.,12.90,0,11865403
3009,OTT0099,Strada OtterBox iPhone Case Leather Folio 6 / ...,Resistant cover card slot for iPhone 6.,49.99,0,11865403
4152,TUC0262,Tucano AL-GO Case iPhone 6 / 6S Gray,Resistant Case for iPhone 6 Plus and 6s,19.90,0,11865403
5227,APP1694,Apple iPhone Leather Case Cover 7 Brown Candy ...,ultrathin leather case and microfiber premium ...,59.00,0,11865403
8360,APP2514,Silicon Case Cover Apple iPhone 8/7 Rosa Arena,Ultrathin silicone case and microfiber premium...,39.00,0,11865403
4524,OTT0132,OtterBox Symmetry Alpha Glass Case + Screen Pr...,Pack OtterBox Symmetry Case + Screen Protector...,49.99,0,11865403
3053,TUC0241,Tucano Tosto Case + Protector iPhone 6 / 6S Gray,Cover for iPhone 6 / 6S.,24.90,0,11865403
1067,MOS0129,Moshi iGlaze Case for iPhone 6 / 6S Rosa,Rigid shell shock protection and rasguÌ ± os f...,30.00,0,11865403


Looks like this is a category of phone cases.

Let's have a look at the 2nd largest type to see if that's also a clear category.

In [ ]:
category_type_df.loc[category_type_df["type"] == "12175397"].sample(10)

,sku,name,desc,price,in_stock,type
3671,PAC1274,Pack QNAP TS-251 | WD 8TB Network,Pack QNAP TS-251 + 8TB (2x4TB) Network WD Hard...,650.89,0,12175397
655,SYN0088,Synology Surveillance Station VS240HD,monitoring station to monitor up to 24 IP came...,453.99,0,12175397
8546,PAC2218,Synology DS718 + NAS Server | 16GB RAM | 8TB (...,Scalable NAS server with transcoding 4K: 4-cor...,980.71,0,12175397
3229,PAC1299,Pack QNAP TS-451 + | 2GB RAM + 8TB Seagate Iro...,Pack QNAP TS-451 + with 2GB RAM memory + 8TB (...,918.95,0,12175397
8477,PAC2260,DS418play Synology NAS Server | 16GB RAM | 32T...,4-bay NAS server to accommodate 4K Ultra HD files,2113.71,0,12175397
3855,PAC1732,QNAP TS-128 Server l Nas 6TB (1x6TB) Seagate I...,NAS TS-128 1 6TB hard drive for Mac and PC,398.98,0,12175397
3347,PAC1316,Pack QNAP TS-253A | 4GB RAM | Seagate 8TB Iron...,QNAP Pack + 4GB memory RAM + 8TB (2x4TB) IronW...,749.97,0,12175397
8523,PAC2379,Synology DS418 NAS Server | 2GB RAM | 16TB (4x...,NAS server 4 bays and 2GB of RAM DDR4 capable ...,1144.95,0,12175397
9412,SYN0182-A,Open - Synology DS118 NAS server 1 bahåÕa,NAS server refitted 1 bay can accommodate hous...,175.99,0,12175397
3261,PAC1292,Pack QNAP TS-251 + | 8GB RAM | Seagate 4TB Iro...,QNAP TS-251 Pack + 8GB RAM + 4TB (2x2TB) IronW...,719.97,0,12175397


Looks like this category is full of servers.

I wonder how many `type`s account for most of our products?

In [ ]:
n = 30
print(f"With the {n} largest types, we account for {((category_type_df.groupby('type').count().nlargest(n, 'sku')['sku'].sum()) / (category_type_df.shape[0]) * 100).round(2)}% of all products.")

With the 30 largest types, we account for 78.4% of all products.


Looks like we can simply investigate 30 types and set the categories, then the remaining 20% of products can have the category `other`.

Use the skills you learnt above to change the category for each type.

In [ ]:
category_type_df['type'] = category_type_df['type'].fillna('no_type')

print(category_type_df['type'].isna().sum())

0


In [ ]:
category_type_df.type.nunique()

126

In [ ]:
# Build a summary table: type code, count, and example product names
type_summary = category_type_df.groupby('type').agg(
    count=('sku', 'count'),
    examples=('name', lambda x: list(x.head(3)))
).reset_index()

# Sort by count, descending, to see the most common types first
type_summary = type_summary.sort_values('count', ascending=False)

# Show all rows (126 types), not just the default preview
pd.set_option('display.max_colwidth', None)
type_summary

,type,count,examples
14,11865403,1057,"[Muvit Back Clear Case iPhone 4 / 4S Transparent, Extreme Muvit iPhone and iPod BikeMount sleeve black Touch, Rollei Chest Mount iPhone 4 / 4S]"
23,12175397,939,"[Sonnet XMAC mini Server, Synology DX513 expansion NAS Mac and PC, QNAP TS-212P NAS server Mac and PC]"
41,1298,783,"[Open - Belkin MIXIT Lightning iPhone Support, Open - Seagate Barracuda 1TB 35 ""SATA 7200rpm hard drive Mac and PC, Open - Western Digital 2TB Green 35 ""5400rpm hard drive Mac and PC]"
17,11935397,562,"[LaCie Porsche Design 1TB External Hard Drive Mac and PC, Envoy OWC USB 3.0 Case for MacBook Air SSD 2010/2011, OWC Case External SuperSlim for SuperDrive MacBook / MacBook Pro]"
16,11905404,454,"[Music Receiver Belkin iPhone music receiver, Withings Wireless Scale Scale iPhone iPad & iPod Touch White, IK Multimedia iRig MIX DJ Mixer iPhone iPad and iPod]"
...,...,...,...
76,21622158,1,[Second hand - Apple Mac mini Core 2 Duo 226Ghz | 4GB RAM | 500GB HDD]
103,51912158,1,"[Open - Apple MacBook Pro 15 ""Core i7 Touch Bar 28GHz | RAM 16GB | 256GB PCIe SSD | 555 2GB Radeon Pro Space Gray]"
102,51902158,1,"[Apple Macbook Air 13 ""i5 16GHz | 8GB RAM | 128GB Flash]"
98,5185,1,[Apple Watch the 1st Gen. 42mm Stainless Steel Case Leather Strap Color Stone L]


In [ ]:
type_to_category = {
    # Laptops
    '1282': 'laptop', '2158': 'laptop', '21632158': 'laptop', '5,39E+11': 'laptop',
    '2,17E+11': 'laptop', '1,02E+12': 'laptop', '51912158': 'laptop', '51902158': 'laptop',
    '9,29E+11': 'laptop',
    # Desktops
    '5,74E+15': 'desktop', '2,16E+11': 'desktop', '118692158': 'desktop',
    '21622158': 'desktop', '5,44E+11': 'desktop', '51882158': 'desktop', '5,43E+15': 'desktop',
    '5,72E+15': 'desktop', '5,45E+15': 'desktop',
    # Smartphones
    '51601716': 'smartphone', '85651716': 'smartphone', '85641716': 'smartphone',
    '24821716': 'smartphone', '24811716': 'smartphone', '113281716': 'smartphone',
    '113291716': 'smartphone', '21561716': 'smartphone', '21571716': 'smartphone',
    '113271716': 'smartphone', '1716': 'smartphone',
    # Tablets
    '12141714': 'tablet', '106431714': 'tablet', '51861714': 'tablet', '1714': 'tablet',
    '13621714': 'tablet', '51871714': 'tablet', '42931714': 'tablet', '24861714': 'tablet',
    '12031714': 'tablet', '113851714': 'tablet', '12051714': 'tablet',
    # iPod
    '11821715': 'ipod', '79201715': 'ipod',
    # Apple Watch
    '24885185': 'apple_watch', '24895185': 'apple_watch', '2449': 'apple_watch',
    '2434': 'apple_watch', '2425': 'apple_watch', '24215399': 'apple_watch', '5185': 'apple_watch',
    # Storage
    '12175397': 'storage', '11935397': 'storage', '1433': 'storage', '12215397': 'storage',
    '57445397': 'storage', '12655397': 'storage', '42945397': 'storage', '1276': 'storage',
    # Cases & Protection
    '11865403': 'case', '12635403': 'case', '13835403': 'case', '13555403': 'case',
    '12575403': 'case', '14035403': 'case',
    # Chargers, Cables & Adapters
    '12585395': 'charger_adapter', '1325': 'charger_adapter', '13005399': 'charger_adapter',
    '5395': 'charger_adapter', '14365395': 'charger_adapter', '13615399': 'charger_adapter',
    '1230': 'charger_adapter',
    # Audio
    '5384': 'audio', '5398': 'audio', '5404': 'audio', '5399': 'audio', '1375': 'audio',
    # Memory (RAM)
    '1364': 'memory',
    # Monitors
    '1296': 'monitor', '1405': 'monitor',
    # Input devices
    '1229': 'input_device', '1387': 'input_device', '13855401': 'input_device',
    '5401': 'input_device', '54025401': 'input_device', '12355400': 'input_device',
    '101781405': 'input_device',
    # Bags & Cases (bulk)
    '1392': 'bag', '10230': 'bag', '5403': 'bag', '5720': 'bag', '1216': 'bag',
    # Repair parts/tools
    '1,44E+11': 'repair_parts', '21485407': 'repair_parts', '21535407': 'repair_parts',
    '54085407': 'repair_parts', '12645406': 'repair_parts', '14305406': 'repair_parts',
    '5406': 'repair_parts', '5407': 'repair_parts', '20642062': 'repair_parts',
    # Software & Services
    '1416': 'software', '1231': 'software', '1424': 'software',
    # Networking
    '1334': 'networking', '9094': 'networking', '1404': 'networking',
    # Open/condition-only (mixed categories)
    '1298': 'open_box_misc',
    # No type
    'no_type': 'no_type',
}

# Apply the mapping; anything not in the dict becomes 'other'
category_type_df['category'] = category_type_df['type'].map(type_to_category).fillna('other')

# Check the result
print(category_type_df['category'].value_counts())

category
storage            2138
case               1807
other               871
open_box_misc       783
laptop              733
charger_adapter     613
desktop             398
audio               372
repair_parts        315
apple_watch         314
monitor             262
smartphone          221
bag                 220
memory              216
input_device        206
networking          196
tablet              176
ipod                 61
no_type              46
software             44
Name: count, dtype: int64


In [ ]:
for cat in category_type_df['category'].unique():
    print(f"\n=== {cat} ===")
    print(category_type_df[category_type_df['category'] == cat]['name'].sample(min(5, len(category_type_df[category_type_df['category'] == cat]))).tolist())


=== other ===
['Hyper Pearl 1600mAh battery Mini USB Mirror and Comic Blond', 'Mophie Juice Pack Plus (2100mAh) battery cover iPhone SE / 5s / 5 Black', 'Parrot Jumping Sumo brown MiniDrone', 'Hyper Pearl 3000mAh USB Battery mirror and Gold', 'Elgato Eve Button button to home automation devices']

=== input_device ===
['Moleskine Smart Writing in September smartpen', 'Wacom Bamboo Stylus duo 3rd Generation Blue', 'Adonith spare ball Adonit Jot Pro / Fip', 'Apple Magic Trackpad 2', 'Wacom Bamboo Stylus Duo 4 Rosa']

=== charger_adapter ===
['Allocacoc PowerCube Original USB Fluorescent lamp White / Red', 'Corning Optical Thunderbolt Cable 10m', 'IAdapt Kanex Mini DisplayPort to HDMI Cable 3m', 'Silver Moshi HDMI-VGA Adapter', 'TP-LINK Cable 1m White Lightning IFM']

=== memory ===
['Mac OWC Memory 1GB 333MHz DDR SO-DIMM', 'Crucial Mac Memory 4GB DDR3 1333MHz SO-DIMM', 'FCM Mac Memory 4GB PC2-5300 DDR2 SO-DIMM 667MHz', 'Mac memory Kingston 32GB (2x16GB) SO-DIMM DDR4 2400MHz', 'Mac Memor

In [ ]:
# Check for other condition-related keywords in product names
keywords = ['Open', 'Second hand', 'Like new', 'Refurbished', 'Used']
for kw in keywords:
    count = category_type_df['name'].str.contains(kw, case=False, na=False).sum()
    print(f"{kw}: {count} products")

Open: 1195 products
Second hand: 91 products
Like new: 340 products
Refurbished: 0 products
Used: 0 products


In [ ]:
def get_condition(name):
    name_lower = str(name).lower()
    if 'second hand' in name_lower:
        return 'second_hand'
    elif 'like new' in name_lower:
        return 'like_new'
    elif 'open' in name_lower:
        return 'open_box'
    else:
        return 'new'

category_type_df['condition'] = category_type_df['name'].apply(get_condition)

print(category_type_df['condition'].value_counts())

condition
new            8391
open_box       1170
like_new        340
second_hand      91
Name: count, dtype: int64


In [ ]:
print(category_type_df['category'].value_counts())

category
storage            2138
case               1807
other               871
open_box_misc       783
laptop              733
charger_adapter     613
desktop             398
audio               372
repair_parts        315
apple_watch         314
monitor             262
smartphone          221
bag                 220
memory              216
input_device        206
networking          196
tablet              176
ipod                 61
no_type              46
software             44
Name: count, dtype: int64


In [ ]:
category_type_df[category_type_df['category'] == 'laptop']['name'].sample(20).tolist()

['Apple Mac mini Core i5 28GHz | 16GB RAM | 256GB Flash (MGEQ2YP / A)',
 'Apple Macbook Pro 15 "Core i7 Touch Bar 31GHz | 16GB | 512GB SSD | Radeon Pro 560 4GB Silver',
 'Apple iMac 27 "Core i5 3.2GHz Retina 5K | 16GB | 512GB Flash',
 'Apple Macbook Pro 13 "Core i5 Touch Bar 33GHz | 16GB | 256GB SSD Space Gray',
 'Apple Macbook Pro 13 "Core i5 2.3GHz | 8GB | 1TB SSD Silver',
 'Like new - Apple Mac mini Core i5 14GHz | 8GB RAM | 500GB',
 'Apple iMac 21.5 "Core i7 33GHz 4K Retina Display | 8GB | Fusion Drive 2TB',
 'Apple MacBook Pro Retina 13 "i5 27 Ghz | RAM 16GB | 128GB Flash',
 'Like new - Apple MacBook Pro Retina 13 "i5 27 GHz | 8GB RAM | 256GB SSD',
 'Apple MacBook Retina 12 "Core M7 13GHz | 8GB RAM | 512GB Gold',
 'Like New - Apple Macbook Pro 13 "2.3GHz i5 Dual-core | 128GB | Space Gray',
 'Apple iMac 215 "Core i5 16GHz | 16GB | 256GB Flash',
 'Apple MacBook Pro 13 "Core i5 with Touch Bar 31GHz | 8GB RAM | 512GB PCIe SSD Plata',
 'Apple Mac mini Core i5 28GHz | 8GB RAM | 1TB Flas

In [ ]:
def fix_laptop_desktop(row):
    name_lower = str(row['name']).lower()

    # Only reclassify if currently in laptop or desktop (categories we know are mixed)
    if row['category'] in ['laptop', 'desktop']:
        if 'macbook' in name_lower:
            return 'laptop'
        elif 'imac' in name_lower or 'mac pro' in name_lower or 'mac mini' in name_lower:
            return 'desktop'
        else:
            return 'other'  # catch anything unexpected that doesn't match either

    return row['category']  # leave everything else untouched

category_type_df['category'] = category_type_df.apply(fix_laptop_desktop, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2138
case               1807
other               875
open_box_misc       783
desktop             675
charger_adapter     613
laptop              452
audio               372
repair_parts        315
apple_watch         314
monitor             262
smartphone          221
bag                 220
memory              216
input_device        206
networking          196
tablet              176
ipod                 61
no_type              46
software             44
Name: count, dtype: int64


In [ ]:
category_type_df[category_type_df['category'] == 'no_type']['name'].tolist()

['SanDisk Cruzer Edge USB 2.0 Flash Drive 16GB',
 'SanDisk Extreme Cruzer 16GB USB 3.0 Flash Drive',
 'Apple iPhone 5S 32GB Space Gray',
 'Apple iPhone 6 16GB Silver',
 'Apple iPhone 6 Plus 16GB Silver',
 'Apple iPhone 6 16GB Space Gray',
 'Apple iPhone 6 64GB Space Gray',
 'Apple iPhone 6 64GB Silver',
 'Apple iPhone 6 Plus 16GB Space Gray',
 'Apple iPhone 6 128GB Gold',
 'Apple iPhone 6 Plus 64GB Space Gray',
 'Apple iPhone 6 Plus 64GB Silver',
 'Apple iPhone 6 Plus 128GB Space Gray',
 'Pack Synology DS1815 + + 40TB WD Red',
 'SanDisk Ultra 16GB SDHC Memory Card Class 10',
 'Apple iPhone 6S 16GB Silver',
 'Elgato Video Capture Mac',
 'Apple iPhone 6S Plus 64GB Rose Gold',
 'Apple iPhone 6S 16GB Space Gray',
 '16GB Apple iPhone 6S Rose Gold',
 'Apple iPhone 6S 64GB Silver',
 'Apple iPhone 6S 64GB Space Gray',
 'Apple iPhone 6S 64GB Gold',
 '64GB Apple iPhone 6S Rose Gold',
 'Apple iPhone 6S Plus 16GB Silver',
 'Apple iPhone 6S Plus 16GB Space Gray',
 'Apple iPhone 6S Plus 16GB Gold',


In [ ]:
def fix_no_type(row):
    if row['category'] != 'no_type':
        return row['category']

    name_lower = str(row['name']).lower()

    if 'iphone' in name_lower:
        return 'smartphone'
    elif 'ipad' in name_lower:
        return 'tablet'
    elif 'mac mini' in name_lower:
        return 'desktop'
    elif any(k in name_lower for k in ['sandisk', 'synology', 'hdd', 'ssd', 'g-drive', 'flash drive', 'memory card']):
        return 'storage'
    elif 'headphone' in name_lower or 'speaker' in name_lower:
        return 'audio'
    else:
        return 'other'

category_type_df['category'] = category_type_df.apply(fix_no_type, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2144
case               1807
other               876
open_box_misc       783
desktop             676
charger_adapter     613
laptop              452
audio               373
repair_parts        315
apple_watch         314
monitor             262
smartphone          257
bag                 220
memory              216
input_device        206
networking          196
tablet              177
ipod                 61
software             44
Name: count, dtype: int64


In [ ]:
category_type_df[category_type_df['category'] == 'open_box_misc']['name'].sample(20).tolist()

['Open - Apple Keyboard Keypad for Mac Spanish',
 '(Open) NewerTech Adapter USB 3.0 to DVI / HDMI / VGA HD',
 'Open - Pebble Smartwatch Time Steel Black',
 'Open - Tangram Smart Led Rope Comba Cromado Size M',
 'Open - Olloclip 4-in-1 Lens iPhone 5 / 5s / SE Gold / White',
 '(Open) iPhone Case OtterBox Symmetry Clear 7 Plus Transparent',
 'Open - Opulus Exo alcantara Case iPhone 5 / 5S Black',
 '(Open) Fitbit Surge Figured Black Clock',
 '(Open) Apple Thunderbolt Display 27 "Monitor Mac',
 'Open - Seagate 8TB HDD Nas IronWolf Sata 3',
 'Open - Memory Crucial Mac 2GB 667MHZ DDR2 SO-DIMM',
 'Open - Kingston V300 SSD Disk 480GB',
 '(Open) Aiino Custodia Elegance iPhone 6 Blue',
 'Open - Satechi USB Hub-C to USB-A / Micro SD / SD / USB-C Charger Silver',
 '(Open) D-Link DAP-1620 Wi-Fi Amplifier AC71200',
 'Open - Belkin Thunderbolt Express Dock 2 HD',
 '(Open) Apple MacBook Pro Retina 13 "i7 3GHz | RAM 16GB | 128GB Flash',
 'Open - WD My Passport Ultra Hard Drive 4TB Mac and PC White',
 '(

In [ ]:
def fix_open_box_misc(row):
    if row['category'] != 'open_box_misc':
        return row['category']

    name_lower = str(row['name']).lower()

    if 'macbook' in name_lower:
        return 'laptop'
    elif 'imac' in name_lower or 'mac pro' in name_lower or 'mac mini' in name_lower:
        return 'desktop'
    elif 'iphone' in name_lower:
        return 'smartphone'
    elif 'ipad' in name_lower:
        return 'tablet'
    elif 'ipod' in name_lower:
        return 'ipod'
    elif 'apple watch' in name_lower or 'watchstand' in name_lower:
        return 'apple_watch'
    elif any(k in name_lower for k in ['hard drive', 'ssd', 'nas', 'flash drive', 'memory card', 'thunderbay', 'raid']):
        return 'storage'
    elif 'case' in name_lower or 'screen protector' in name_lower or 'protector' in name_lower:
        return 'case'
    elif any(k in name_lower for k in ['adapter', 'cable', 'charger', 'dock', 'hub']):
        return 'charger_adapter'
    elif 'headphone' in name_lower or 'speaker' in name_lower:
        return 'audio'
    elif 'monitor' in name_lower or 'display' in name_lower:
        return 'monitor'
    elif 'keyboard' in name_lower or 'mouse' in name_lower or 'stylus' in name_lower:
        return 'input_device'
    elif 'router' in name_lower or 'repeater' in name_lower or 'wi-fi' in name_lower:
        return 'networking'
    elif 'ram' in name_lower or 'memory' in name_lower:
        return 'memory'
    else:
        return 'other'

category_type_df['category'] = category_type_df.apply(fix_open_box_misc, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2253
case               1811
other              1021
desktop             707
charger_adapter     681
laptop              562
smartphone          392
audio               389
apple_watch         334
repair_parts        315
monitor             311
memory              235
tablet              225
input_device        224
bag                 220
networking          206
ipod                 62
software             44
Name: count, dtype: int64


In [ ]:
category_type_df[category_type_df['category'] == 'other']['name'].sample(30).tolist()

['Pure Sport Armband iPhone Rosa',
 'Mophie Juice Pack Helium (1500mAh) battery cover iPhone SE / 5s / 5 red',
 '(Open) Sonnet SuperSpeed \u200b\u200bUSB 3.0 ExpressCard / 34',
 'Hue Philips Living Colors Bloom Iris in September + Bridge',
 'Elgato Eve Button button to home automation devices',
 'Kensington Lift Off MacBook Support',
 'Wahoo Fitness BlueSC cadence and speed sensor',
 'Promise Pegasus R4 Edition 3 Symply 12TB Hard Disk Thunderbolt 3',
 'MyFox Security Camera for Home Security Alarm System',
 'Withings Body Scale Black',
 'IK Multimedia iRig Stomp Guitar Pedal iPhone iPad and iPod',
 'Open - Sandisk iXpand Lightning to USB 3.0 64GB',
 'Rain Design iLevel2 MacBook Support',
 'Withings Smart Steel HR 36 Black Clock',
 'Open - Seagate 2TB hard disk FireCuda Hibrido SSHD 35 Sata 3',
 '(Open) Wacom Intuos Creative Art Pen & Touch M Black',
 'Philips Hue White Ambiance Pack 2 bulbs E27 Starter Kit + Bridge + Dimmer',
 'Installation Kit OWC SSD / HDD 25 "Mac mini (2011/2012)',


In [ ]:
def refine_other(row):
    if row['category'] != 'other':
        return row['category']

    name_lower = str(row['name']).lower()

    if any(k in name_lower for k in ['power bank', 'power reserve', 'power pack', 'power capsule', 'external battery']):
        return 'power_bank'
    elif any(k in name_lower for k in ['smart plug', 'philips hue', 'tado', 'netatmo', 'smart climate', 'lightstrip']):
        return 'smart_home'
    elif any(k in name_lower for k in ['fitbit', 'pebble']):
        return 'wearable_other'
    elif any(k in name_lower for k in ['drone', 'gopro', 'vr virtual reality', 'action cam']):
        return 'gadget'
    else:
        return 'other'

category_type_df['category'] = category_type_df.apply(refine_other, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2253
case               1811
other               835
desktop             707
charger_adapter     681
laptop              562
smartphone          392
audio               389
apple_watch         334
repair_parts        315
monitor             311
memory              235
tablet              225
input_device        224
bag                 220
networking          206
ipod                 62
smart_home           56
power_bank           55
software             44
gadget               44
wearable_other       31
Name: count, dtype: int64


In [ ]:
category_type_df.sample(30)

,sku,name,desc,price,in_stock,type,category,condition
3066,ICA0066,"Incase Neoprene Sleeve Case Classic Macbook Pro / Air / Retina 13 ""Black",Flexibe light cover and neoprene.,39.95,0,13835403,case,new
8530,PAC2202,Synology DS718 + NAS Server | 2GB RAM | 6TB (2x3TB) WD Red,Scalable NAS server and transcoding 4K: 4-core 2.3 GHz 2GB DDR3L 226MB / s read and 184 MB / s write,717.99,0,12175397,storage,new
419,IKM0022,IK Multimedia iKlip two microphone adapter mini black IPAD,IPad mini universal stand microphone stand.,28.99,1,1216,bag,new
119,APP0490,Apple Airport Express,Airport Express base station 802.11a / b / g / n.,109.00,0,1334,networking,new
9219,ICA0092,"Incase Compact Backpack City MacBook 15 ""Gray",Incase backpack with 175 liters of cargo compartment 15 inch MacBook and iPhone pocket,99.95,1,1392,bag,new
9515,APP1529-A,"Open - Apple iPad Silicone Case Pro 97 ""Lavender",Reconditioned sleeve lightweight silicone and soft touch for iPad Pro 97-inch,79.00,0,12635403,case,open_box
7696,STA0020-A,Open - Startech Adapter Lightning to Micro USB-B Female White,Lightning to Micro USB Adapter for iPhone-B generation iPad and iPod Touch 6A.,16.99,0,1298,charger_adapter,open_box
396,TRI0012,Star Wars Chewbacca Tribe 8GB USB 2.0 Pen Drive,PenDrive 8GB USB 2.0 + 1GB online edition Star Wars.,19.99,0,11935397,storage,new
3226,PAC1394,Pack QNAP TS-451 + | 8GB RAM | WD 32TB Network,Pack QNAP TS-451 + with 8GB of RAM memory + 32TB (4x8TB) Network WD Hard Drive for Mac and PC.,1935.99,0,12175397,storage,new
4321,APP1595,Space Gray Apple Watch Sport Black Woven Nylon Strap 38mm,Space Gray Apple Watch Sport 38mm black nylon strap,369.00,0,24885185,apple_watch,new


In [ ]:
category_type_df[category_type_df['category'] == 'smartphone'].sample(30)

,sku,name,desc,price,in_stock,type,category,condition
6853,APP2024,Open - Apple iPhone 5s 16GB Space Gray - As new,Free Apple iPhone 5s 16GB Space Gray (ME432Y / A),489.00,0,1716,smartphone,open_box
7866,LIF0113-A,Open - LifeProof Nood Submersible iPhone Case 7 Black,waterproof and resistant to extreme conditions Case for iPhone 7,89.99,0,1298,smartphone,open_box
6000,BEL0277-A,Open - Belkin Sport Armband Pro-Fit Bracelet Black iPhone 7,lightweight neoprene armband with reflective tape and large belt loop for iPhone 7,34.99,0,1298,smartphone,open_box
5995,GRT0430-A,Open - Journey Griffin Survivor iPhone Case 7 Black / Pink,Cast and impact resistant padded case for iPhone 7,29.99,0,1298,smartphone,open_box
7139,AP20190,Open - Apple iPhone 5S 16GB Silver,Apple iPhone 5s 16GB refitted Free Color Silver,409.00,0,51601716,smartphone,open_box
8222,APP2496,Apple iPhone 7 Plus 32GB Black Bright,New Apple iPhone 7 Plus 32GB Black Free Bright,779.00,0,85651716,smartphone,new
6001,APP1688-A,(Open) Apple iPhone Leather Case Cover 7 Brown Candy Plus,ultrathin leather case and microfiber premium for iPhone 7 Plus,59.00,0,1298,smartphone,open_box
5159,APP1648,Apple iPhone 7 128GB Black,New Apple iPhone 7 Free Black 128GB,749.00,0,85641716,smartphone,new
6433,MOS0154-A,Open - Moshi iVisorGlass Protector iPhone 6 / 6S White,Screen saver superfine glass for iPhone 6 / 6S,29.99,0,1298,smartphone,open_box
7126,AP20103,Like new - Apple iPhone 64GB Space Gray,Apple iPhone SE Free Refurbished 64GB Space Gray,549.00,0,51601716,smartphone,like_new


In [ ]:
def classify_iphone_accessory(name):
    name_lower = str(name).lower()

    # Real phones: no accessory keyword present
    accessory_keywords = [
        'case', 'support', 'protector', 'cover', 'charger', 'cable', 'dock',
        'battery', 'lens', 'kit', 'glass', 'screen', 'armband', 'stand',
        'mount', 'adapter', 'keyboard', 'holder', 'sleeve', 'bag', 'strap',
        'headset', 'microphone', 'pointer', 'plug', 'connector', 'part',
        'piece', 'repair', 'tools', 'shield', 'skin', 'protect', 'pod',
        'control', 'oxómetro', 'pulse', 'pressure', 'monopod', 'trigger'
    ]

    if not any(kw in name_lower for kw in accessory_keywords):
        return 'smartphone'

    # Specific accessory sub-categories, checked in order of priority
    if any(kw in name_lower for kw in ['case', 'cover', 'sleeve', 'bag', 'protector', 'shield', 'screen', 'glass', 'skin', 'protect']):
        return 'case'
    elif any(kw in name_lower for kw in ['charger', 'cable', 'dock', 'adapter', 'plug', 'connector']):
        return 'charger_adapter'
    elif any(kw in name_lower for kw in ['battery', 'pod']):
        return 'power_bank'
    elif any(kw in name_lower for kw in ['support', 'stand', 'mount', 'holder', 'armband', 'strap']):
        return 'bag'  # mounts/stands/straps grouped with accessories
    elif any(kw in name_lower for kw in ['keyboard']):
        return 'input_device'
    elif any(kw in name_lower for kw in ['microphone', 'headset']):
        return 'audio'
    elif any(kw in name_lower for kw in ['plug', 'smart']):
        return 'smart_home'
    elif any(kw in name_lower for kw in ['repair', 'part', 'piece', 'tools']):
        return 'repair_parts'
    elif any(kw in name_lower for kw in ['pointer', 'control', 'lens']):
        return 'other'
    else:
        return 'other'

def fix_smartphone(row):
    if row['category'] != 'smartphone':
        return row['category']
    return classify_iphone_accessory(row['name'])

category_type_df['category'] = category_type_df.apply(fix_smartphone, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2253
case               1904
other               839
desktop             707
charger_adapter     687
laptop              562
audio               391
apple_watch         334
repair_parts        318
monitor             311
smartphone          263
memory              235
bag                 231
input_device        226
tablet              225
networking          206
power_bank           63
ipod                 62
smart_home           56
software             44
gadget               44
wearable_other       31
Name: count, dtype: int64


In [ ]:
# --- 1. apple_watch: split real watches vs third-party accessories ---
acc_brands = ['griffin','x-doria','nomad','twelve south','belkin','zagg','panama','sena','cygnett',
              'elago','just mobile','incase','hoco','satechi','epik','lunatik','native union',
              'mophie','switcheasy','kosta bluelounge','elevation lab','force band','moxie','speck',
              'otterbox','uag','urban armor']

def classify_watch(name):
    n = str(name).lower()
    if any(b in n for b in acc_brands):
        return 'watch_accessory'
    if 'case' in n:  # real watches mention a case material (aluminum/steel case)
        return 'apple_watch'
    return 'watch_accessory'  # straps/docks without case mention

mask = category_type_df['category'] == 'apple_watch'
category_type_df.loc[mask, 'category'] = category_type_df.loc[mask, 'name'].apply(classify_watch)

# --- 2. monitor: pull out graphics tablets, fitness "activity monitors", baby monitors, adapters ---
def fix_monitor(row):
    if row['category'] != 'monitor':
        return row['category']
    n = str(row['name']).lower()
    if 'wacom' in n:
        return 'graphics_tablet'
    if 'activity monitor' in n or 'jawbone' in n:
        return 'other'
    if 'baby monitor' in n or 'babycam' in n:
        return 'other'
    if 'displayport to' in n or 'mini displayport' in n or 'adapter' in n:
        return 'charger_adapter'
    if 'calibrator' in n:
        return 'other'
    return 'monitor'

category_type_df['category'] = category_type_df.apply(fix_monitor, axis=1)

# --- 3. software: move optical drives/recorders (hardware, not software) to storage ---
def fix_software(row):
    if row['category'] != 'software':
        return row['category']
    n = str(row['name']).lower()
    if any(k in n for k in ['recorder', 'dvd', 'bluray', 'blu-ray', 'writer', 'superdrive', 'nvr']):
        return 'storage'
    return 'software'

category_type_df['category'] = category_type_df.apply(fix_software, axis=1)

# --- 4. power_bank: move non-battery accessories out ---
def fix_power_bank(row):
    if row['category'] != 'power_bank':
        return row['category']
    n = str(row['name']).lower()
    if 'stand' in n:
        return 'other'
    if 'pointer' in n:
        return 'input_device'
    if 'mic' in n or 'microphone' in n:
        return 'audio'
    if 'selfie' in n:
        return 'other'
    if 'blood pressure' in n:
        return 'other'
    if 'controller' in n and 'keys' in n:
        return 'input_device'
    if 'battery kit' in n or 'change' in n:
        return 'repair_parts'
    return 'power_bank'

category_type_df['category'] = category_type_df.apply(fix_power_bank, axis=1)

# --- 5. bag: split cases, batteries, and stands out; keep only true bags/backpacks ---
def fix_bag(row):
    if row['category'] != 'bag':
        return row['category']
    n = str(row['name']).lower()
    bag_kw = ['backpack', 'briefcase', 'bandolier', 'sleeve', 'maletin', 'maleta', 'mochila']
    if any(k in n for k in bag_kw):
        return 'bag'
    if 'external battery' in n:
        return 'power_bank'
    if 'case' in n or 'cover' in n:
        return 'case'
    if 'hirise' in n or 'stand' in n or 'support' in n or 'mount' in n or 'holder' in n or 'kiosk' in n or 'clamp' in n or 'enclosure' in n:
        return 'other'
    return 'bag'

category_type_df['category'] = category_type_df.apply(fix_bag, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2263
case               1936
other               983
desktop             707
charger_adapter     690
laptop              562
audio               394
repair_parts        319
smartphone          263
memory              235
input_device        228
monitor             228
tablet              225
watch_accessory     220
networking          206
apple_watch         114
bag                  95
graphics_tablet      73
ipod                 62
smart_home           56
power_bank           55
gadget               44
software             34
Name: count, dtype: int64


In [ ]:
category_type_df.loc[category_type_df['category'] == 'wearable_other', 'category'] = 'other'

print(category_type_df['category'].value_counts())

category
storage            2253
case               1904
other               870
desktop             707
charger_adapter     687
laptop              562
audio               391
apple_watch         334
repair_parts        318
monitor             311
smartphone          263
memory              235
bag                 231
input_device        226
tablet              225
networking          206
power_bank           63
ipod                 62
smart_home           56
software             44
gadget               44
Name: count, dtype: int64


In [ ]:
# Rename categories for clarity
category_type_df['category'] = category_type_df['category'].replace({
    'gadget': 'tech_gadgets',
    'bag': 'bags_backpacks'
})

# Sweep ALL categories for case/cover/folio/sleeve/protector keywords, moving them to 'case'
# Exclude categories where these words are legitimate and NOT meant as protective cases:
#   - bags_backpacks (real bags), case (already there), storage (has "enclosure"),
#   - apple_watch (real watches legitimately contain "case" = case material)
case_kw = ['case', 'cover', 'folio', 'sleeve', 'protector']
exclude_from_sweep = ['bags_backpacks', 'case', 'storage', 'apple_watch']

def sweep_case(row):
    if row['category'] in exclude_from_sweep:
        return row['category']
    n = str(row['name']).lower()
    if any(k in n for k in case_kw):
        if 'keyboard' in n:
            return 'keyboards'
        return 'case'
    return row['category']

category_type_df['category'] = category_type_df.apply(sweep_case, axis=1)

# Pull standalone keyboards (no case keyword) out of input_device into keyboards
def pull_keyboards(row):
    if row['category'] != 'input_device':
        return row['category']
    n = str(row['name']).lower()
    if 'keyboard' in n:
        return 'keyboards'
    return 'input_device'

category_type_df['category'] = category_type_df.apply(pull_keyboards, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2263
case               2128
other               911
desktop             707
charger_adapter     689
laptop              534
audio               389
repair_parts        309
smartphone          263
memory              235
monitor             228
networking          198
tablet              192
watch_accessory     188
input_device        180
apple_watch         114
bags_backpacks       95
graphics_tablet      69
ipod                 62
smart_home           56
keyboards            52
power_bank           52
tech_gadgets         44
software             34
Name: count, dtype: int64


In [ ]:
category_type_df[category_type_df['category'] == 'smart_home'].sample(30)

,sku,name,desc,price,in_stock,type,category,condition
3403,PHI0068,Philips HUE LightStrips Plus LEDs Strip 1m extension,LEDS strip with maximum flexibility and 1 meter in length,24.95,1,11905404,smart_home,new
3395,PHI0061,Philips HUE Bridge 2.0 for lamps and bulbs Hue,Bridge intelligent control of lights and accessories compatible with HomeKit Hue,59.95,1,11905404,smart_home,new
5475,PHI0077,Philips Hue Single Bulb E27 White Ambiance,Individual E27 different color temperatures Philips HUE,34.95,1,11905404,smart_home,new
9537,KOO0007,Koogeek Homekit Smart Plug socket with Siri voice control,Koogeek plug with Apple HomeKit technology and Siri voice control for your home,34.99,0,11905404,smart_home,new
6135,PHI0066-A,Open - Philips Hue White Set 2 bulbs 9.5W E27 + A60 Bridge,Kit 2 + bridge white bulbs illuminated HUE,79.95,0,1298,smart_home,open_box
6537,PHI0081,Philips HUE Single Bulb GU10 White Ambience,Bulb with different shades of white for iPhone iPad and iPod,34.95,1,11905404,smart_home,new
2277,PHI0057,Philips HUE Lux A19 9W E27 set of 2 bulbs + Bridge,HUE Starter Kit: Pack of Hue LED Light Bulbs + Bridge Lux 9W for iPhone iPad and iPod Touch.,99.95,0,11905404,smart_home,new
3404,PHI0069,Philips Hue White 9.5W E27 A60 Single + Dimmer Switch,Independent bulb set white + remote control for lamps and bulbs Hue,39.95,1,11905404,smart_home,new
1626,PHI0050,Philips Hue LightStrips independent LED strip,LED strip Hue expansion.,89.99,0,11905404,smart_home,new
3490,NEA0001-A,(Open) Netatmo Urban weather station,Wireless weather station for iPhone iPad or iPod Touch.,169.99,0,1298,smart_home,open_box


In [ ]:
# --- Merge input_device + keyboards into one unified category ---
category_type_df['category'] = category_type_df['category'].replace({
    'keyboards': 'input_devices',
    'input_device': 'input_devices'
})

# --- Fix charger_adapter stragglers ---
def fix_charger(row):
    if row['category'] != 'charger_adapter':
        return row['category']
    n = str(row['name']).lower()
    if 'bike rack' in n or 'quickmount' in n:
        return 'other'
    if 'battery' in n and ('macbook' in n or 'powerbook' in n or 'imac' in n):
        return 'repair_parts'
    return 'charger_adapter'

category_type_df['category'] = category_type_df.apply(fix_charger, axis=1)

# --- Fix networking stragglers ---
def fix_networking(row):
    if row['category'] != 'networking':
        return row['category']
    n = str(row['name']).lower()
    if 'ram memory' in n or ('memory' in n and 'ddr' in n):
        return 'memory'
    if 'warranty' in n:
        return 'other'
    return 'networking'

category_type_df['category'] = category_type_df.apply(fix_networking, axis=1)

print(category_type_df['category'].value_counts())

category
storage            2263
case               2128
other               914
desktop             707
charger_adapter     676
laptop              534
audio               389
repair_parts        320
smartphone          263
memory              238
input_devices       232
monitor             228
networking          194
tablet              192
watch_accessory     188
apple_watch         114
bags_backpacks       95
graphics_tablet      69
ipod                 62
smart_home           56
power_bank           52
tech_gadgets         44
software             34
Name: count, dtype: int64


In [ ]:
category_type_df['category'] = category_type_df['category'].replace({'tablet': 'ipad'})

print(category_type_df['category'].value_counts())

category
storage            2263
case               2128
other               914
desktop             707
charger_adapter     676
laptop              534
audio               389
repair_parts        320
smartphone          263
memory              238
input_devices       232
monitor             228
networking          194
ipad                192
watch_accessory     188
apple_watch         114
bags_backpacks       95
graphics_tablet      69
ipod                 62
smart_home           56
power_bank           52
tech_gadgets         44
software             34
Name: count, dtype: int64


In [ ]:
def fix_other(row):
    if row['category'] != 'other':
        return row['category']
    n = str(row['name']).lower()

    fitness_kw = ['fitbit','withings','jawbone','scale smart','smart scale']
    gadget_kw = ['sphero']
    stand_kw = ['stand','support','mount','holder','kiosk','clamp','enclosure','dock']

    if any(k in n for k in fitness_kw):
        return 'fitness_wearable'
    if any(k in n for k in gadget_kw):
        return 'tech_gadgets'
    if any(k in n for k in stand_kw):
        return 'stands_mounts'
    return 'other'

category_type_df['category'] = category_type_df.apply(fix_other, axis=1)

print(category_type_df['category'].value_counts())

category
storage             2263
case                2128
desktop              707
charger_adapter      676
other                619
laptop               534
audio                389
repair_parts         320
smartphone           263
memory               238
input_devices        232
monitor              228
stands_mounts        208
networking           194
ipad                 192
watch_accessory      188
apple_watch          114
bags_backpacks        95
graphics_tablet       69
fitness_wearable      66
tech_gadgets          65
ipod                  62
smart_home            56
power_bank            52
software              34
Name: count, dtype: int64


In [ ]:
# Fix 1: GoPro accessories belong with other action-camera gadgets, not input devices
mask1 = (category_type_df['category']=='input_devices') & (category_type_df['name'].str.lower().str.contains('gopro'))
category_type_df.loc[mask1, 'category'] = 'tech_gadgets'

# Fix 2: consolidate all Nest products into smart_home (was split across other/networking/stands_mounts)
mask2 = category_type_df['name'].str.lower().str.contains('nest ')
category_type_df.loc[mask2, 'category'] = 'smart_home'

# Fix 3: Twelve South HiRise stands belong in stands_mounts
mask3 = (category_type_df['category']=='other') & (category_type_df['name'].str.lower().str.contains('hirise'))
category_type_df.loc[mask3, 'category'] = 'stands_mounts'

print(category_type_df['category'].value_counts())

category
storage             2263
case                2128
desktop              707
charger_adapter      676
other                612
laptop               534
audio                389
repair_parts         320
smartphone           263
memory               238
monitor              228
input_devices        226
stands_mounts        212
ipad                 192
networking           191
watch_accessory      188
apple_watch          114
bags_backpacks        95
tech_gadgets          71
graphics_tablet       69
fitness_wearable      66
ipod                  62
smart_home            62
power_bank            52
software              34
Name: count, dtype: int64


In [ ]:
# ============================================================
# CONSOLIDATED FIXES - Exhaustive audit session
# ============================================================

# --- 1. Storage enclosures/cases misclassified as generic "case" ---
storage_fixes = [
    'Akitio Thunder2 QUAD Case External Thunderbolt 2',
    'Akitio NODE Case External Thunderbolt PCIe Graphics Card 3',
    'Open - Akitio NODE Case External Thunderbolt PCIe Graphics Card 3',
    'Akitio Thunder3 QUAD X Case External Thunderbolt 3',
    'Satechi External Case 25 "Aluminum HDD and SSD USB-C Plata',
    'Open - Satechi External Case 25 "Aluminum HDD and SSD USB-C Plata',
    'Satechi External Case 25 "Aluminum HDD and SSD USB-C Gray Space',
    'LMP DataPort External Enclosure 2.5 "USB 3.0',
    'LMP DataStore External Enclosure 35 "USB 3.0 Aluminum',
    'OWC Express External Enclosure 25 "USB 3.0 Plata',
    'OWC Express External Enclosure 25 "USB 3.0 White',
    'OWC Express USB 3.0 External Enclosure 25 Black',
    'Macally USB 3.0 External Enclosure 2.5 "SATA Plata',
    'Macally USB 3.0 External Enclosure 3.5 "SATA Plata',
    'Open - Macally USB 3.0 External Enclosure 3.5 "SATA Plata',
    'Open - Macally USB 3.0 External Enclosure 2.5 "SATA Plata',
    'Open - LMP DataPort External Enclosure 2.5 "USB 3.0',
]
for name in storage_fixes:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'storage'

# --- 2. Networking extenders miscaught by "coverage" substring match on "cover" ---
networking_fixes = [
    'TP-Link RE450 coverage Extender - repeater Wi-Fi AC1750',
    'TP-Link Extender coverage RE210 - AC750 Wi-Fi repeater',
    'TP-Link Extender coverage RE200 - AC750 Wi-Fi repeater',
    'Open - TP-Link Extender coverage RE200 - AC750 Wi-Fi repeater',
    'TP-Link RE305 coverage Extender - repeater Wi-Fi AC1200',
    'Open - TP-Link RE450 coverage Extender - repeater Wi-Fi AC1750',
    'Open - TP-Link Extender coverage RE210 - AC750 Wi-Fi repeater',
]
for name in networking_fixes:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'networking'
category_type_df.loc[category_type_df['name']=='Synology RAM 4GB 1866MHz DDR3', 'category'] = 'memory'
category_type_df.loc[category_type_df['name']=='Nokia Home Air Quality Sensor HD Camera', 'category'] = 'smart_home'

# --- 3. Repair parts mislabeled as "cases" (mistranslation of "carcasa") ---
repair_fixes = [
    'repair battery cases (1st generation)', 'Connector repair charge cases (1st generation)',
    'posterior chamber repair cases 2', 'Part iFixit battery cases mini 2/3',
    'Repair cases mini load connector 3', 'Connector mini load repair cases 2',
    'Internal Speaker repair cases Mini 2', 'Speaker mini repair lower cases 3',
    'OWC Hard Drive Support 3.5 "Anti Mac Pro 2009-2012',
]
for name in repair_fixes:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'repair_parts'

# --- 4. Power banks / UPS mislabeled ---
power_bank_fixes = [
    'SurgeCube Belkin Surge Protector + 2 USB',  # actually goes to charger_adapter, see below
]
category_type_df.loc[category_type_df['name']=='SurgeCube Belkin Surge Protector + 2 USB', 'category'] = 'charger_adapter'
for name in ['PFC Sinewave CyberPower 900VA 540W UPS System', 'PFC Sinewave CyberPower 1300VA 780W UPS System',
             'PFC Sinewave CyberPower 1500VA 900W UPS System',
             'Mophie Power Reserve micro USB External Battery 1350mAh Black',
             'Mophie Battery 1400mAh Power Capsule micro USB Adapter']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'power_bank'

# --- 5. Stands/mounts scattered across other categories ---
stands_fixes = [
    'Hitcase ChestR Chest Mount Support', 'Hitcase SuckR Car Mount Holder', 'Hitcase TubulR Mount Support Bar',
    'iOttie Easy Flex 3 Car Car Support iPhone White',
    'iOttie Easy One Touch XL Car Support iPhone 6 and iPhone 6 Plus',
    'Moxie Pinza 3M Car Support Compatible with all iPhone',
    'Moxie Car Support for compatible iPhone Volante',
    'Just Mobile headstand headset support', 'H-Squared Air Mount for Airport Express 2012',
    'Maclocks Security Security Support Apple TV Mount',
    'BlueLounge Kickflip stand MacBook Pro 13 "Black', 'Macally Bamboo Cooling Stand Macbook',
    'Wacom Cintiq Ergo Stand', 'Wacom Cintiq Ergo Stand 27QHD',
    'IK Multimedia iKlip microphone black iPad 2 Adapter',
    'IK Multimedia iKlip two microphone adapter mini black IPAD',
    'Griffin Multidock February 10 bays Cargo Security and iPad',
    'iOttie Easy One Touch 2 iPhone', 'Runtastic universal bike fixing',
    'Twelve South Compass iPad 2 Silver', 'Ik Multimedia iKlip A / V iPhone',
    'Kenu compact tripod Lightning Stance Orange iPhone',
    'elago H Headphones Silver Stand Support', 'elago H Headphones Support Stand Gray Space',
    'Satechi Support Aluminum Silver Headphones',
    'Satechi Support Aluminum Headphones with 3 USB / Jack 3.5 Plata',
    'Satechi Support Aluminum Earphones with 3 USB / Jack 3.5 Space Gray',
    'Satechi Support Aluminum Gray Space Headphones',
    '(Open) Satechi Support Aluminum Silver Headphones',
    'iOttie One Touch Easy Qi Wireless Charger iPhone Support X / 8 Plus / 8',
    'Cavus Play 1 Black Wall Support', 'Cavus Support Wall Play 1 White',
    'Cavus Foot Support Sonos Play 1 Black', 'Cavus Foot Support Sonos Play 1 White',
]
for name in stands_fixes:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'stands_mounts'
category_type_df.loc[category_type_df['name'].str.contains('HiRise.*Duet|Henge Docks Clique', case=False, regex=True, na=False), 'category'] = 'stands_mounts'
category_type_df.loc[category_type_df['name']=='BlueLounge Jimi iMac USB expansion', 'category'] = 'charger_adapter'

# --- 6. Calibrators / screen cleaner -> other ---
for name in ['Moshi TeraGlove cleaner screens', 'X-Rite ColorMunki Photo Calibrator',
             'X-Rite ColorMunki Photo + GrafiLite Free', 'X-Rite ColorMunki Display monitor calibrator',
             'Wacom Color Calibrator Manager Mac and PC', 'X-Rite i1 Display monitor calibrator PRO',
             'X-Rite color calibrator i1Studio', 'Elgato Game Capture video recorder HD60S game',
             'Wikango 600 Spanish warning radars', 'Wikango 700 Spanish warning radars battery',
             'Tangram Smart Led Rope Comba Cromado Size M']:
    category_type_df.loc[category_type_df['name'].str.contains(name, case=False, regex=False, na=False), 'category'] = 'other'

# --- 7. Monitor: graphics tablets (Wacom Paper) + real monitors w/ speakers ---
category_type_df.loc[(category_type_df['category']=='monitor') & (category_type_df['name'].str.contains('Intuos Pro Paper|Paper Pro Intuos', case=False, na=False)), 'category'] = 'graphics_tablet'
for name in ['Open - BenQ EW2775ZH VA Panel Monitor 27 Slim Speakers 2Wx2',
             'Open - LG 34UM68-P Monitor 34 "IPS HDMI Speakers WFHD',
             'Open - LG 43UD79-B Monitor 425 "4K 72% NTSC USB-C Speakers DisplayPort']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'monitor'

# --- 8. Audio: Qi chargers, smart home lamp, e-scooter ---
for name in ['Stacked Wall Charger Wireless Charger White wall', 'Minibatt iCharger Qi Wireless Charger Quick charge',
             'Belkin Base Qi Wireless charging up Boost iPhone 75W X / 8/8 Plus',
             'Hoco Wireless Charging Pad Qi Wireless Charger 5V / 1A Black',
             'Hoco Wireless Charging Pad Qi Wireless Charger 5V / 1A White',
             'Devia Qi Wireless Charger Black 9v', 'Qi Wireless Charger Fast 9v Devia White',
             'Ryval Base Magnetic Qi Wireless Charging Office', 'Ryval magnetic Qi Wireless Car Charger',
             'MiniBatt Qi Wireless Charging Receiver Card']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'charger_adapter'
for name in ['Elgato Avea Bluetooth Sphere Lamp', 'Open - Elgato avea Sphere Lamp']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'smart_home'
for name in ['Ninebot by Segway Personal Transportation Robot miniPRO White',
             'Robot miniPRO Ninebot by Segway personal transport two wheels Black',
             'Open - Ninebot by Segway Personal Transportation Robot miniPRO two wheels Black']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'tech_gadgets'

# --- 9. Bags: mounts, cases, armbands, GoPro accessory ---
for name in ['IK Multimedia iKlip microphone black iPad 2 Adapter']:
    pass  # already handled above
for name in ['Moshi iGlaze Armor iPhone 6 / 6S Rose Gold', 'Moshi iGlaze Armor iPhone 6 / 6S Plus Rose Gold']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'case'
for name in ['selfie extendable arm with black Bluetooth button',
             'Open - Belkin Sport Armband Pro-Fit Bracelet Black iPhone 7',
             'Open - Adidas armband Griffin iPhone 7 Plus / 6s Plus / 6 Plus Black / Red']:
    category_type_df.loc[category_type_df['name']==name, 'category'] = 'other'
category_type_df.loc[category_type_df['name']=='Floaty GoPro Hero & Hero 5 Session Session', 'category'] = 'tech_gadgets'

# --- 10. Sphero robot fix (from earlier session) ---
category_type_df.loc[(category_type_df['category']=='fitness_wearable') & (category_type_df['name'].str.contains('sphero', case=False, na=False)), 'category'] = 'tech_gadgets'

print(category_type_df['category'].value_counts())
print("Total:", len(category_type_df))

category
storage             2280
case                2104
desktop              707
charger_adapter      679
other                626
laptop               534
audio                353
repair_parts         329
smartphone           263
memory               238
stands_mounts        238
monitor              226
input_devices        211
networking           196
ipad                 192
watch_accessory      188
apple_watch          114
bags_backpacks        81
tech_gadgets          75
graphics_tablet       74
fitness_wearable      66
smart_home            65
ipod                  62
power_bank            57
software              34
Name: count, dtype: int64
Total: 9992


In [ ]:
from google.colab import files

category_type_df.to_csv('category_type_df.csv', index=False)
files.download('category_type_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Opción directa (evalúa la condición booleana y obtiene la media)
porcentaje = (category_type_df['category'] == 'other').mean() * 100
print(f"El porcentaje de 'other' es: {porcentaje:.2f}%")

El porcentaje de 'other' es: 6.27%
